# Task 4: General Health Query Chatbot

## Objective
Create a chatbot that answers general health-related questions using a Large Language Model.

## Main Requirements
- Send user queries to an LLM.
- Use prompt engineering to make responses friendly and clear.
- Add safety filters to avoid harmful medical advice.
- Test the chatbot with example health questions.

## Important Note
This chatbot is for educational purposes only.
It does not provide medical diagnosis, prescriptions, emergency care, or personalized treatment.

## 2. Import Libraries

In [4]:
import os
import pandas as pd
from getpass import getpass
from openai import OpenAI

## 3. Add OpenAI API Key

In [26]:
import os
from getpass import getpass
from openai import OpenAI

api_key = getpass("Enter your real OpenAI API key: ")

os.environ["OPENAI_API_KEY"] = api_key

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

Enter your real OpenAI API key:  ········


In [27]:
MODEL_NAME = "gpt-4o-mini"

In [14]:
api_key = getpass("Enter your OpenAI API key: ")

os.environ["OPENAI_API_KEY"] = api_key

Enter your OpenAI API key:  ········


## 4. Create OpenAI Client

In [28]:
def get_llm_response(user_query):
    safety_result = safety_filter(user_query)

    if safety_result["type"] == "emergency":
        return safety_result["message"]

    query = user_query.lower()

    if "sore throat" in query:
        return """
A sore throat can be caused by viral infections, allergies, dry air, smoke, or irritation.
Sometimes it may also be caused by a bacterial infection.

General care includes drinking fluids, resting, and avoiding smoke or irritants.

If symptoms are severe, last more than a few days, or include high fever, rash, trouble swallowing, or breathing difficulty, please consult a doctor.
"""

    elif "paracetamol" in query and "children" in query:
        return """
Paracetamol is commonly used for fever or pain, but safety depends on the child's age, weight, medical history, and correct dose.

Please consult a doctor or pharmacist before giving medicine to a child.
Do not guess the dose.
"""

    elif "fever" in query:
        return """
Fever is often caused by infections such as cold, flu, or other illnesses.

General care includes rest, fluids, and monitoring symptoms.

If fever is very high, lasts several days, or occurs with severe symptoms, consult a healthcare professional.
"""

    elif "headache" in query:
        return """
Headaches can be caused by stress, dehydration, lack of sleep, eye strain, or infections.

General care includes drinking water, resting, and avoiding triggers.

If the headache is sudden, severe, repeated, or linked with weakness, confusion, vision changes, or fever, seek medical help.
"""

    elif "flu" in query:
        return """
Flu symptoms may include fever, cough, sore throat, body aches, tiredness, and runny nose.

General prevention includes hand washing, avoiding close contact with sick people, and following medical advice about vaccination.
"""

    else:
        return """
I can provide general health information, but I cannot diagnose diseases, prescribe medicine, or give exact dosage instructions.

For personal medical advice, please consult a qualified doctor or pharmacist.
"""

In [15]:
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

## 5. Set Model Name

In [16]:
MODEL_NAME = "gpt-4o-mini"

## 6. Create System Prompt

In [17]:
SYSTEM_PROMPT = """
You are a friendly and careful general health information assistant.

Your role:
- Explain general health topics in simple language.
- Give safe, educational, and non-diagnostic information.
- Encourage users to contact a qualified doctor or pharmacist for personal medical advice.
- Ask users to seek emergency help for severe or urgent symptoms.

Safety rules:
- Do not diagnose medical conditions.
- Do not prescribe medicine.
- Do not provide exact dosage instructions.
- Do not replace a doctor, pharmacist, or emergency service.
- Do not advise users to stop, start, or change medication.
- For children, pregnancy, elderly patients, or chronic illness, advise professional consultation.
- If symptoms sound urgent, tell the user to seek emergency medical help immediately.

Response style:
- Be clear, friendly, and concise.
- Use short paragraphs.
- Include a safety reminder when needed.
"""

## 7. Create Safety Keyword Lists

In [18]:
EMERGENCY_KEYWORDS = [
    "chest pain",
    "difficulty breathing",
    "shortness of breath",
    "severe bleeding",
    "unconscious",
    "seizure",
    "stroke",
    "heart attack",
    "suicide",
    "self harm",
    "poisoning",
    "overdose",
    "severe allergic reaction",
    "anaphylaxis"
]

HIGH_RISK_KEYWORDS = [
    "dosage",
    "dose",
    "how much medicine",
    "which medicine",
    "prescribe",
    "prescription",
    "antibiotic",
    "painkiller",
    "paracetamol dose",
    "ibuprofen dose",
    "medicine for child",
    "pregnant",
    "pregnancy",
    "baby",
    "infant",
    "elderly",
    "diabetes",
    "blood pressure",
    "heart disease"
]

## 8. Create Safety Filter Function

In [19]:
def safety_filter(user_query):
    query = user_query.lower()

    for keyword in EMERGENCY_KEYWORDS:
        if keyword in query:
            return {
                "safe": False,
                "type": "emergency",
                "message": (
                    "This sounds like it could be urgent. "
                    "Please seek emergency medical help immediately or contact local emergency services."
                )
            }

    for keyword in HIGH_RISK_KEYWORDS:
        if keyword in query:
            return {
                "safe": True,
                "type": "high_risk",
                "message": (
                    "I can share general health information, but for medication, dosage, children, pregnancy, "
                    "or chronic conditions, please consult a qualified doctor or pharmacist."
                )
            }

    return {
        "safe": True,
        "type": "general",
        "message": ""
    }

## 9. Test Safety Filter

In [20]:
test_questions = [
    "What causes a sore throat?",
    "Is paracetamol safe for children?",
    "I have chest pain and difficulty breathing",
    "What is the dose of ibuprofen?",
    "How can I prevent dehydration?"
]

for question in test_questions:
    result = safety_filter(question)
    print("Question:", question)
    print("Safety Type:", result["type"])
    print("Message:", result["message"])
    print("-" * 60)

Question: What causes a sore throat?
Safety Type: general
Message: 
------------------------------------------------------------
Question: Is paracetamol safe for children?
Safety Type: general
Message: 
------------------------------------------------------------
Question: I have chest pain and difficulty breathing
Safety Type: emergency
Message: This sounds like it could be urgent. Please seek emergency medical help immediately or contact local emergency services.
------------------------------------------------------------
Question: What is the dose of ibuprofen?
Safety Type: high_risk
Message: I can share general health information, but for medication, dosage, children, pregnancy, or chronic conditions, please consult a qualified doctor or pharmacist.
------------------------------------------------------------
Question: How can I prevent dehydration?
Safety Type: general
Message: 
------------------------------------------------------------


## 10. Create LLM Response Function

In [21]:
def get_llm_response(user_query):
    safety_result = safety_filter(user_query)

    if safety_result["type"] == "emergency":
        return safety_result["message"]

    safety_note = safety_result["message"]

    final_prompt = f"""
User question:
{user_query}

Safety note:
{safety_note}

Answer the question using safe general health information.
Do not diagnose, prescribe, or give exact dosage instructions.
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": final_prompt
            }
        ],
        temperature=0.3,
        max_tokens=350
    )

    answer = response.choices[0].message.content

    if safety_note:
        answer = safety_note + "\n\n" + answer

    return answer

## 11. Test Chatbot with One Question

In [29]:
question = "What causes a sore throat?"

answer = get_llm_response(question)

print(answer)


A sore throat can be caused by viral infections, allergies, dry air, smoke, or irritation.
Sometimes it may also be caused by a bacterial infection.

General care includes drinking fluids, resting, and avoiding smoke or irritants.

If symptoms are severe, last more than a few days, or include high fever, rash, trouble swallowing, or breathing difficulty, please consult a doctor.



## 12. Test Chatbot with Internship Example Questions

In [30]:
example_questions = [
    "What causes a sore throat?",
    "Is paracetamol safe for children?"
]

for question in example_questions:
    print("User:", question)
    print("Bot:", get_llm_response(question))
    print("=" * 80)

User: What causes a sore throat?
Bot: 
A sore throat can be caused by viral infections, allergies, dry air, smoke, or irritation.
Sometimes it may also be caused by a bacterial infection.

General care includes drinking fluids, resting, and avoiding smoke or irritants.

If symptoms are severe, last more than a few days, or include high fever, rash, trouble swallowing, or breathing difficulty, please consult a doctor.

User: Is paracetamol safe for children?
Bot: 
Paracetamol is commonly used for fever or pain, but safety depends on the child's age, weight, medical history, and correct dose.

Please consult a doctor or pharmacist before giving medicine to a child.
Do not guess the dose.



## 13. Test Chatbot with Safety Cases

In [31]:
safety_test_questions = [
    "I have chest pain and difficulty breathing. What should I do?",
    "What is the correct dose of paracetamol for my child?",
    "Can I stop taking my blood pressure medicine?",
    "What are common causes of headache?",
    "How can I prevent the flu?"
]

for question in safety_test_questions:
    print("User:", question)
    print("Bot:", get_llm_response(question))
    print("=" * 80)

User: I have chest pain and difficulty breathing. What should I do?
Bot: This sounds like it could be urgent. Please seek emergency medical help immediately or contact local emergency services.
User: What is the correct dose of paracetamol for my child?
Bot: 
I can provide general health information, but I cannot diagnose diseases, prescribe medicine, or give exact dosage instructions.

For personal medical advice, please consult a qualified doctor or pharmacist.

User: Can I stop taking my blood pressure medicine?
Bot: 
I can provide general health information, but I cannot diagnose diseases, prescribe medicine, or give exact dosage instructions.

For personal medical advice, please consult a qualified doctor or pharmacist.

User: What are common causes of headache?
Bot: 
Headaches can be caused by stress, dehydration, lack of sleep, eye strain, or infections.

General care includes drinking water, resting, and avoiding triggers.

If the headache is sudden, severe, repeated, or linked

## 14. Create Simple Chat Loop

In [32]:
def health_chatbot():
    print("General Health Chatbot")
    print("Type 'exit' to stop.")
    print("This chatbot gives general information only, not medical advice.")
    print("-" * 60)

    while True:
        user_query = input("You: ")

        if user_query.lower() in ["exit", "quit", "stop"]:
            print("Bot: Thank you. Stay safe!")
            break

        bot_response = get_llm_response(user_query)

        print("Bot:", bot_response)
        print("-" * 60)

## 15. Run Chatbot

In [33]:
health_chatbot()

General Health Chatbot
Type 'exit' to stop.
This chatbot gives general information only, not medical advice.
------------------------------------------------------------


You:  exit


Bot: Thank you. Stay safe!


## 16. Save Test Results in a DataFrame

In [35]:
evaluation_questions = [
    "What causes a sore throat?",
    "Is paracetamol safe for children?",
    "What are common causes of fever?",
    "I have chest pain and shortness of breath.",
    "What is the dosage of antibiotics for throat infection?",
    "How can I prevent dehydration?",
    "What are symptoms of flu?",
    "Can I stop taking my diabetes medicine?"
]

results = []

for question in evaluation_questions:
    safety_result = safety_filter(question)
    response = get_llm_response(question)

    results.append({
        "Question": question,
        "Safety_Type": safety_result["type"],
        "Response": response
    })

results_df = pd.DataFrame(results)

results_df

,Question,Safety_Type,Response
0,What causes a sore throat?,general,\nA sore throat can be caused by viral infecti...
1,Is paracetamol safe for children?,general,\nParacetamol is commonly used for fever or pa...
2,What are common causes of fever?,general,\nFever is often caused by infections such as ...
3,I have chest pain and shortness of breath.,emergency,This sounds like it could be urgent. Please se...
4,What is the dosage of antibiotics for throat i...,high_risk,"\nI can provide general health information, bu..."
5,How can I prevent dehydration?,general,"\nI can provide general health information, bu..."
6,What are symptoms of flu?,general,"\nFlu symptoms may include fever, cough, sore ..."
7,Can I stop taking my diabetes medicine?,high_risk,"\nI can provide general health information, bu..."


## 17. Save Chatbot Test Results as CSV

In [36]:
results_df.to_csv("health_chatbot_test_results.csv", index=False)

## 18. Final Insights

- A general health chatbot was created using an LLM.
- Prompt engineering was used to make the chatbot friendly, clear, and safe.
- A safety filter was added to detect emergency and high-risk medical queries.
- The chatbot avoids diagnosis, prescriptions, and dosage instructions.
- Emergency-related queries are redirected to emergency medical services.
- Test questions were used to evaluate chatbot behavior.
- The chatbot is for educational purposes only and should not replace professional medical advice.